In [ ]:
# !pip install minio delta-spark

In [ ]:
import os
from myutils import remove_location
from minio import Minio
from pyspark.sql import SparkSession
# Importar as funções necessárias
from pyspark.sql import functions as F
from delta.tables import *

MINIO_ACCESS_KEY = "pkIeKAn4xpjoOgXiHQPw"
MINIO_SECRET_KEY = "JZRBfRszxLZzPeaAQaEXk32JxUKv25DVjUoO06Rk"
BUCKET_BRONZE = "bronze"
BUCKET_SILVER = "silver"
BUCKET_GOLD = "gold"

In [ ]:
%%time
spark = SparkSession.builder.master("spark://spark-master:7077") \
    .appName("MyAppAula02") \
    .config("spark.eventLog.enabled", "true") \
    .config("spark.eventLog.dir", "file:/tmp/spark-logs") \
    .config("spark.history.fs.logDirectory", "file:/tmp/spark-logs") \
    .config("log4j.rootCategory", "INFO, console") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "1536m") \
    .config("spark.driver.memory", "1536m") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.storage.memoryFraction", "0.4") \
    .config("spark.shuffle.memoryFraction", "0.5") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "512m") \
    .config("spark.sql.parquet.compression.codec", "gzip") \
    .config("spark.sql.orc.compression.codec", "zlib") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC") \
    .getOrCreate()

In [ ]:
sc = spark.sparkContext
hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
hadoop_conf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

In [ ]:
spark

## PROJECT 1: HOTEL BOOKING

- [Data source](https://www.kaggle.com/datasets/mojtaba142/hotel-booking)

Let's explore the data, also creating an index using z-Ordering and cache strategy

1. Reading data from raw from **Staging**
2. Write data into bronze
3. Explore columns and make discoveries which will be useful to analyze
4. Masking Sensitive Data (PII) with hash
5. Write data into **Silver**
6. Choice the index column (by preference a primary ID column) - **Silver**
7. Making analytics with the data calculating some important metrics, to consume to the Dataviz (Dashboard) later
8. Write data into **Gold**
9. Now, apply cache in the **Gold** table to enhance the performance of every consult

### 1 - READING STAGING (RAW)

In [ ]:
location_raw = f"s3a://staging"
file = "hotel_booking.csv"
data_origen = f"{location_raw}/{file}"

In [ ]:
df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(data_origen)

### 2 - Write data into bronze

In [ ]:
table = "hotel_booking_bronze"

In [ ]:
location_bronze = f"s3a://{BUCKET_BRONZE}/delta/{table}"

In [ ]:
# Writing in Delta format
df.write.format("delta") \
    .mode("overwrite") \
    .save(location_bronze)

### 3 - Explore columns and make discoveries which will be useful to analyze

In [ ]:
df_delta = spark.read.format('delta').load(location_bronze)

In [ ]:
len(df_delta.columns)

In [ ]:
%%time
df_delta.count()

In [ ]:
df_delta.printSchema()

### 4 - Masking Sensitive Data (PII) with hash
The query selects only the columns relevant for analysis, masking sensitive personal information such as name, email, phone number, and credit card.

In [ ]:
# Masking a column 'name' using SHA-256
df_delta = df_delta.withColumn('name', F.sha2(F.col('name'), 256))

# Masking a column 'email' using SHA-256
df_delta = df_delta.withColumn('email', F.sha2(F.col('email'), 256))

# Masking a column 'phone-number' using SHA-256
df_delta = df_delta.withColumn('phone-number', F.sha2(F.col('phone-number'), 256))

# Masking a column 'credit_card' using SHA-256
df_delta = df_delta.withColumn('credit_card', F.sha2(F.col('credit_card'), 256))

### 5 - Write data into silver

In [ ]:
table = "hotel_booking_silver"
location_silver = f"s3a://{BUCKET_SILVER}/delta/{table}"

In [ ]:
# Writing in Delta format
df_delta.write.format("delta") \
    .mode("overwrite") \
    .save(location_silver)

### 6 - Choice the index column (by preference a primary ID column) - Silver

In [ ]:
df_silver = spark.read.format('delta').load(location_silver)

In [ ]:
df_silver.createOrReplaceTempView("hotel_bookings_silver")

In [ ]:
spark.sql("""
    SELECT
      COUNT(DISTINCT name) AS unique_name_values,
      COUNT(DISTINCT email) AS unique_email_values,
      COUNT(DISTINCT credit_card) AS unique_credit_card_values,
      COUNT(DISTINCT hotel) AS unique_hotel_values,
      COUNT(DISTINCT is_canceled) AS unique_is_canceled_values,
      COUNT(DISTINCT lead_time) AS unique_lead_time_values,
      COUNT(DISTINCT arrival_date_year) AS unique_arrival_date_year_values,
      COUNT(DISTINCT arrival_date_month) AS unique_arrival_date_month_values,
      COUNT(DISTINCT arrival_date_week_number) AS unique_arrival_date_week_number_values,
      COUNT(DISTINCT arrival_date_day_of_month) AS unique_arrival_date_day_of_month_values,
      COUNT(DISTINCT stays_in_weekend_nights) AS unique_stays_in_weekend_nights_values,
      COUNT(DISTINCT stays_in_week_nights) AS unique_stays_in_week_nights_values,
      COUNT(DISTINCT adults) AS unique_adults_values,
      COUNT(DISTINCT children) AS unique_children_values,
      COUNT(DISTINCT babies) AS unique_babies_values,
      COUNT(DISTINCT meal) AS unique_meal_values,
      COUNT(DISTINCT country) AS unique_country_values,
      COUNT(DISTINCT market_segment) AS unique_market_segment_values,
      COUNT(DISTINCT distribution_channel) AS unique_distribution_channel_values,
      COUNT(DISTINCT is_repeated_guest) AS unique_is_repeated_guest_values,
      COUNT(DISTINCT previous_cancellations) AS unique_previous_cancellations_values,
      COUNT(DISTINCT previous_bookings_not_canceled) AS unique_previous_bookings_not_canceled_values,
      COUNT(DISTINCT reserved_room_type) AS unique_reserved_room_type_values,
      COUNT(DISTINCT assigned_room_type) AS unique_assigned_room_type_values,
      COUNT(DISTINCT booking_changes) AS unique_booking_changes_values,
      COUNT(DISTINCT deposit_type) AS unique_deposit_type_values,
      COUNT(DISTINCT agent) AS unique_agent_values,
      COUNT(DISTINCT company) AS unique_company_values,
      COUNT(DISTINCT days_in_waiting_list) AS unique_days_in_waiting_list_values,
      COUNT(DISTINCT customer_type) AS unique_customer_type_values,
      COUNT(DISTINCT adr) AS unique_adr_values,
      COUNT(DISTINCT required_car_parking_spaces) AS unique_required_car_parking_spaces_values,
      COUNT(DISTINCT total_of_special_requests) AS unique_total_of_special_requests_values,
      COUNT(DISTINCT reservation_status) AS unique_reservation_status_values,
      COUNT(DISTINCT reservation_status_date) AS unique_reservation_status_date_values
    FROM hotel_bookings_silver;
""").show()

In [ ]:
spark.sql("""
    SELECT
        COUNT(email) as total
    FROM hotel_bookings_silver;
""").limit(10).show(truncate=False)

In [ ]:
# Executar o comando OPTIMIZE com ZORDER BY
spark.sql(f"""
    OPTIMIZE delta.`{location_silver}`
    ZORDER BY (email)
""")

In [ ]:
# Executar o comando OPTIMIZE com ZORDER BY
spark.sql(f"""
    OPTIMIZE delta.`{location_silver}`
    ZORDER BY (reservation_status_date)
""")

In [ ]:
# Create a DeltaTable object
delta_table = DeltaTable.forPath(spark, location_silver)
# Get history of table
history_df = delta_table.history()
history_df.select(["operation", "operationMetrics"]).show(truncate=False)

### 6 - Making analytics with the data calculating some important metrics, to consume to the Dataviz (Dashboard) later

In [ ]:
table = "hotel_booking_gold"
location_silver = f"s3a://{BUCKET_SILVER}/delta/{table}"

In [ ]:
spark.sql(f"""
    SELECT
        COUNT(*)
    FROM hotel_bookings
    WHERE
        LOWER(reservation_status) = 'canceled'
""").limit(10).toPandas().tail()

In [ ]:
spark.stop()